# QGNN

In [ ]:
#download
%pip install -q \
    kagglehub \
    pennylane \
    torch-geometric \
    datasets \
    networkx \
    polars \
    dataframes \
    fastapi \
    wandb
    
%pip install -q pennylane-lightning-gpu

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## Import IBM dataset (Polars lazy scan + 1M / ~4% k-hop subsample)

Primary window first. k-hop seeds are **train-period laundering** only (same recipe as `EDA-IBM.ipynb`). Cache: `ibm_hi_medium_sample.parquet` — reused only if `seed_policy == train_fraud_v1`, height ≥ 1M, and mix ~4%.


In [ ]:
# Polars lazy scan of HI-Medium + k-hop neighborhood subsample (same 1M / ~4% recipe as EDA-IBM).
import os
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 110

MIN_ROWS = 1_000_000
MIN_FRAUD_RATE = 0.04
RANDOM_STATE = 42
MAX_HOPS = 4
MAX_POOL_ACCOUNTS = 800_000
SAMPLE_PATH = Path("ibm_hi_medium_sample.parquet")
TEST_CSV_PATH = Path("ibm_hi_medium_test.csv")
SEED_POLICY = "train_fraud_v1"
MP_POLICY = "split_prefix_v1"
PRIMARY_START = datetime(2022, 9, 1)
PRIMARY_END = datetime(2022, 9, 16, 23, 59)

KAGGLE_N_TX = 32_000_000
KAGGLE_N_LAUNDER = 35_000
KAGGLE_N_ACCOUNTS = 2_077_000
KAGGLE_RATE = 1 / 905

TARGET_AUPRC = (0.70, 0.80)
LEAK_AUPRC = 0.82
PATIENCE = 4

RATES = {
    "US Dollar": 1.4,
    "Euro": 1.64,
    "Bitcoin": 90000.0,
    "Yuan": 0.2,
    "Yen": 0.0067,
    "UK Pound": 2.0,
    "Brazil Real": 0.2,
    "Australian Dollar": 1.0,
    "Rupee": 0.01,
    "Ruble": 0.01,
    "Canadian Dollar": 1.25,
    "Mexican Peso": 0.07,
    "Swiss Franc": 1.5,
    "Shekel": 0.25,
    "Saudi Riyal": 0.25,
}

_SCAN_I = 0


class _Pipe:
    """Lightweight stand-in so ported EDA cells can call pipe.section(...)."""

    def section(self, name):
        print("\n" + "=" * 64, flush=True)
        print(name, flush=True)
        print("=" * 64, flush=True)


pipe = _Pipe()


def annotate_bars(ax, fmt="{:,.0f}", as_pct=False):
    for p in ax.patches:
        h = p.get_height()
        label = f"{h:.2f}%" if as_pct else fmt.format(h)
        if h == 0:
            continue
        ax.text(p.get_x() + p.get_width() / 2, h, label, ha="center", va="bottom", fontsize=9)
    ax.margins(y=0.18)


def collect_scan(lazy, desc=None) -> pl.DataFrame:
    """Collect a lazy query; prefer streaming so full-file scans do not blow RAM."""
    global _SCAN_I
    _SCAN_I += 1
    label = desc or f"Polars scan #{_SCAN_I}"
    t0 = time.perf_counter()
    print(f"  >> {label} ...", flush=True)
    if isinstance(lazy, pl.DataFrame):
        out = lazy
    else:
        try:
            out = lazy.collect(engine="streaming")
        except Exception:
            out = lazy.collect()
    dt = time.perf_counter() - t0
    print(f"  << {label}  ({dt:.1f}s, {out.height:,} rows)", flush=True)
    return out


def try_import_kagglehub():
    try:
        import kagglehub
        return kagglehub
    except ImportError:
        import subprocess, sys
        print("kagglehub not installed — installing it into this kernel...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
        import kagglehub
        return kagglehub


def resolve_ibm_data_dir():
    attached = Path("/kaggle/input/ibm-transactions-for-anti-money-laundering-aml")
    if attached.is_dir() and (attached / "HI-Medium_Trans.csv").is_file():
        return str(attached)
    kagglehub = try_import_kagglehub()
    return kagglehub.dataset_download(
        "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
    )


DATA_DIR = resolve_ibm_data_dir()
TRANS_FILE = os.path.join(DATA_DIR, "HI-Medium_Trans.csv")

print("DATA_DIR:", DATA_DIR)
print("Files:", sorted(os.listdir(DATA_DIR)))
print("Trans exists:", os.path.isfile(TRANS_FILE))

lf_raw = pl.scan_csv(TRANS_FILE)
if "Account_duplicated_0" in lf_raw.collect_schema().names():
    lf_raw = lf_raw.rename({"Account_duplicated_0": "Account.1"})
print("Schema:")
print(lf_raw.collect_schema())


### Schema preview and published-table check


In [ ]:
# --- one scan: row count, label mix, nulls, timestamp span ---
overview = collect_scan(
    lf_raw.select(
        pl.len().alias("n_rows"),
        pl.col("Is Laundering").sum().alias("n_laundering"),
        pl.col("Is Laundering").mean().alias("laundering_rate"),
        pl.col("Timestamp").min().alias("ts_min_raw"),
        pl.col("Timestamp").max().alias("ts_max_raw"),
    ),
    desc="full-file row count + laundering rate (first full-file pass)",
)
print(overview)
print()
n_rows = int(overview["n_rows"][0])
n_laundering = int(overview["n_laundering"][0])
full_rate = float(overview["laundering_rate"][0])
print(f"Measured rows:        {n_rows:,}   (Kaggle ~{KAGGLE_N_TX:,})")
print(f"Measured laundering:  {n_laundering:,}   (Kaggle ~{KAGGLE_N_LAUNDER:,})")
print(f"Measured rate:        {full_rate:.6f}  ({full_rate*100:.4f}%)")
print(f"Kaggle 1/905:         {KAGGLE_RATE:.6f}  ({KAGGLE_RATE*100:.4f}%)")
print(f"1-in-N:               {n_rows / max(n_laundering, 1):.1f}")

print("\nHead (5 rows from the scan):")
display(collect_scan(lf_raw.head(5)))

print("\nNull counts (should be ~0):")
display(collect_scan(lf_raw.select(pl.all().null_count())))


## 2. Tabular EDA on the full file (lazy)

Quality, class imbalance, Payment Format, FX, cardinality, and the post-window artifact. Dummy AUPRC equals prevalence — that is the metric we train on.


In [ ]:
# --- quality: negatives, key-wise duplicates, amount describe on same-currency rows ---
pipe.section("2 Tabular EDA")

if "Account_duplicated_0" in lf_raw.collect_schema().names():
    lf_raw = lf_raw.rename({"Account_duplicated_0": "Account.1"})

amt_quality = collect_scan(
    lf_raw.select(
        (pl.col("Amount Paid") < 0).sum().alias("neg_paid"),
        (pl.col("Amount Received") < 0).sum().alias("neg_received"),
        (pl.col("Amount Paid") == 0).sum().alias("zero_paid"),
        (pl.col("Amount Received") == 0).sum().alias("zero_received"),
        (pl.col("Amount Paid") == pl.col("Amount Received")).mean().alias("pct_paid_eq_received"),
        (pl.col("Payment Currency") == pl.col("Receiving Currency")).mean().alias("pct_same_currency"),
    )
)
print("Amount / currency quality:")
print(amt_quality)

print("\nAmount Received describe (same-currency rows only — mixed FX is not comparable):")
same_ccy = lf_raw.filter(pl.col("Payment Currency") == pl.col("Receiving Currency"))
_amt_cols = ["Amount Received", "Amount Paid"]
_amt_stats = ["count", "null_count", "mean", "std", "min", "25%", "50%", "75%", "max"]
_amt_wide = collect_scan(
    same_ccy.select(
        *[pl.col(c).count().cast(pl.Float64).alias(f"{c}__count") for c in _amt_cols],
        *[pl.col(c).null_count().cast(pl.Float64).alias(f"{c}__null_count") for c in _amt_cols],
        *[pl.col(c).mean().alias(f"{c}__mean") for c in _amt_cols],
        *[pl.col(c).std().alias(f"{c}__std") for c in _amt_cols],
        *[pl.col(c).min().alias(f"{c}__min") for c in _amt_cols],
        *[pl.col(c).quantile(0.25).alias(f"{c}__25%") for c in _amt_cols],
        *[pl.col(c).median().alias(f"{c}__50%") for c in _amt_cols],
        *[pl.col(c).quantile(0.75).alias(f"{c}__75%") for c in _amt_cols],
        *[pl.col(c).max().alias(f"{c}__max") for c in _amt_cols],
    ),
    desc="amount describe (same-currency)",
)
print(
    pl.DataFrame(
        {
            "statistic": _amt_stats,
            **{c: [_amt_wide[f"{c}__{s}"][0] for s in _amt_stats] for c in _amt_cols},
        }
    )
)

dup_stats = collect_scan(
    lf_raw.group_by(
        [
            "Timestamp",
            "From Bank",
            "Account",
            "To Bank",
            "Account.1",
            "Amount Paid",
            "Amount Received",
            "Payment Format",
        ]
    )
    .agg(pl.len().alias("n"))
    .select(
        (pl.col("n") > 1).sum().alias("n_duplicate_groups"),
        pl.col("n").filter(pl.col("n") > 1).sum().alias("n_rows_in_duplicate_groups"),
    )
)
print("\nDuplicate groups on (time, accounts, banks, amounts, format):")
print(dup_stats)


In [ ]:
# --- label bar + dummy baselines (AUPRC is the metric we will train on) ---
print("Is Laundering counts:")
print(f"  0 (legit):       {n_rows - n_laundering:,}")
print(f"  1 (laundering):  {n_laundering:,}")
print(f"  full-file rate:  {full_rate*100:.4f}%   vs Kaggle 1/905 = {KAGGLE_RATE*100:.4f}%")
print("\nA ~4% / 1M-row training subsample is an UPSAMPLE of this ~0.110% prior, not the real world rate.")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    ["not laundering (0)", "laundering (1)"],
    [n_rows - n_laundering, n_laundering],
    color=["steelblue", "crimson"],
)
ax.set_title("HI-Medium label mix (full file)")
ax.set_ylabel("transactions")
annotate_bars(ax)
plt.tight_layout()
plt.show()

print("\n--- Dummy predictors from full-file counts ---")
print(f"Accuracy always-legit: {1 - full_rate:.6f}")
print(f"Accuracy always-fraud: {full_rate:.6f}")
print("Precision/recall always-legit: 0 / 0  (never predicts a positive)")
print(f"Precision/recall always-fraud: {full_rate:.6f} / 1.0")
print(f"AUPRC of a constant score equals prevalence ≈ {full_rate:.6f}")
print("(Use AUPRC for training; accuracy is meaningless at ~0.110% positives.)")

probe_n = min(500_000, n_rows)
pos_n = max(1, int(probe_n * full_rate))
neg_n = probe_n - pos_n
probe = pl.concat(
    [
        collect_scan(lf_raw.filter(pl.col("Is Laundering") == 1).select("Is Laundering").head(pos_n)),
        collect_scan(lf_raw.filter(pl.col("Is Laundering") == 0).select("Is Laundering").head(neg_n)),
    ]
)
y = probe["Is Laundering"].to_numpy()
zeros = np.zeros_like(y)
ones = np.ones_like(y)
print("\n--- sklearn on a ~500k stratified probe ---")
print("Accuracy always-legit:", accuracy_score(y, zeros))
print("AUPRC always-legit scores (all 0.0):", average_precision_score(y, zeros))
print("AUPRC always-fraud scores (all 1.0):", average_precision_score(y, ones))
print("Precision always-legit:", precision_score(y, zeros, zero_division=0))
print("Precision always-fraud:", precision_score(y, ones, zero_division=0))
print("Recall always-legit:", recall_score(y, zeros, zero_division=0))
print("Recall always-fraud:", recall_score(y, ones, zero_division=0))


In [ ]:
# --- Payment Format: IBM analogue of PaySim's fraud-by-type bar ---
fmt = collect_scan(
    lf_raw.group_by("Payment Format").agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").sum().alias("n_laundering"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("n_laundering", descending=True)
)
print(fmt)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(fmt["Payment Format"].to_list(), fmt["n_laundering"].to_list(), color="crimson")
ax.set_title("Laundering count by Payment Format (full file)")
ax.set_ylabel("laundering transactions")
ax.tick_params(axis="x", rotation=30)
annotate_bars(ax)
plt.tight_layout()
plt.show()


In [ ]:
# --- currencies, paid vs received mismatches (FX / conversion edges) ---
print("Receiving Currency:")
print(collect_scan(lf_raw.group_by("Receiving Currency").len().sort("len", descending=True)))
print("\nPayment Currency:")
print(collect_scan(lf_raw.group_by("Payment Currency").len().sort("len", descending=True)))

mismatch = collect_scan(
    lf_raw.select(
        (pl.col("Payment Currency") != pl.col("Receiving Currency")).mean().alias("pct_cross_currency"),
        (pl.col("Amount Paid") != pl.col("Amount Received")).mean().alias("pct_amount_mismatch"),
        (
            (pl.col("Amount Paid") != pl.col("Amount Received"))
            & (pl.col("Payment Currency") == pl.col("Receiving Currency"))
        ).mean().alias("pct_mismatch_same_ccy"),
    )
)
print("\nFX / amount mismatch rates:")
print(mismatch)


In [ ]:
# --- cardinality: banks, accounts, overlap, self-transfers ---
card = collect_scan(
    lf_raw.select(
        pl.col("From Bank").n_unique().alias("n_from_bank"),
        pl.col("To Bank").n_unique().alias("n_to_bank"),
        pl.col("Account").n_unique().alias("n_account"),
        pl.col("Account.1").n_unique().alias("n_account1"),
        pl.col("Payment Format").n_unique().alias("n_format"),
        (
            (pl.col("Account") == pl.col("Account.1"))
            & (pl.col("From Bank") == pl.col("To Bank"))
        ).mean().alias("pct_self_transfer"),
        (pl.col("From Bank") != pl.col("To Bank")).mean().alias("pct_cross_bank"),
    )
)
print(card)
print(f"\nKaggle account count ~{KAGGLE_N_ACCOUNTS:,} (union of senders+receivers, not Account n_unique alone)")

from_banks = collect_scan(lf_raw.select(pl.col("From Bank").unique().alias("b")))
to_banks = collect_scan(lf_raw.select(pl.col("To Bank").unique().alias("b")))
bank_overlap = from_banks.join(to_banks, on="b", how="inner").height
print(
    f"Bank overlap: {bank_overlap:,}  "
    f"({100*bank_overlap/max(from_banks.height,1):.1f}% of From, "
    f"{100*bank_overlap/max(to_banks.height,1):.1f}% of To)"
)


### Timestamps and the post-window artifact


In [ ]:
# --- parse timestamps; count the laundering tail after 16 Sep 2022 ---
lf_ts = lf_raw.with_columns(
    pl.col("Timestamp").str.to_datetime("%Y/%m/%d %H:%M", strict=False).alias("ts")
)
ts_span = collect_scan(
    lf_ts.select(
        pl.col("ts").min().alias("ts_min"),
        pl.col("ts").max().alias("ts_max"),
        pl.col("ts").dt.date().n_unique().alias("n_calendar_days"),
        pl.col("ts").is_null().sum().alias("n_parse_fail"),
    )
)
print(ts_span)

post = lf_ts.filter(pl.col("ts") > pl.lit(PRIMARY_END))
post_stats = collect_scan(
    post.select(
        pl.len().alias("n_post_window"),
        pl.col("Is Laundering").mean().alias("post_laundering_rate"),
        pl.col("Is Laundering").sum().alias("n_post_laundering"),
    )
)
print("\nPost-primary-window tail (do NOT use as test — label mix is broken):")
print(post_stats)

in_window = lf_ts.filter(
    (pl.col("ts") >= pl.lit(PRIMARY_START)) & (pl.col("ts") <= pl.lit(PRIMARY_END))
)
win_stats = collect_scan(
    in_window.select(
        pl.len().alias("n_primary_window"),
        pl.col("Is Laundering").sum().alias("n_laundering"),
        pl.col("Is Laundering").mean().alias("rate"),
    )
)
print("\nInside primary window Sep 1 – Sep 16:")
print(win_stats)


## 3. Fraud-pattern EDA — what concentrates laundering?

These lifts justify the 11-d edge recipe: Payment Format one-hot, scaled amount, time-of-window, self-transfer / cross-bank / cross-currency flags. We do **not** one-hot banks.


In [ ]:
# --- Payment Format LIFT: rate, not just count (GNN should one-hot this) ---
pipe.section("3 Fraud patterns")
fmt = fmt.with_columns((pl.col("rate") / full_rate).alias("lift_vs_prior"))
print(fmt)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(fmt["Payment Format"].to_list(), (fmt["rate"] * 100).to_list(), color="darkred")
ax.axhline(full_rate * 100, color="steelblue", linestyle="--", label=f"full-file prior {full_rate*100:.3f}%")
ax.set_title("Laundering *rate* by Payment Format")
ax.set_ylabel("laundering rate (%)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
annotate_bars(ax, as_pct=True)
plt.tight_layout()
plt.show()


In [ ]:
# --- amount shift: log-amount, laundering vs legit (stratified probe; mixed FX skipped) ---
n_amt = 100_000
amt_pos = collect_scan(
    same_ccy.filter(pl.col("Is Laundering") == 1)
    .select(pl.col("Amount Paid").alias("amount"), pl.col("Is Laundering"))
    .head(n_amt)
)
amt_neg = collect_scan(
    same_ccy.filter(pl.col("Is Laundering") == 0)
    .select(pl.col("Amount Paid").alias("amount"), pl.col("Is Laundering"))
    .head(n_amt)
)
amt_probe = pl.concat([amt_pos, amt_neg]).with_columns(pl.col("amount").log1p().alias("log1p_amount"))
print("Same-currency Amount Paid by label (probe):")
print(amt_probe.group_by("Is Laundering").agg(
    pl.col("amount").median().alias("median"),
    pl.col("amount").mean().alias("mean"),
    pl.col("amount").quantile(0.9).alias("p90"),
))

fig, ax = plt.subplots(figsize=(8, 4))
for lab, color, name in [(0, "steelblue", "legit"), (1, "crimson", "laundering")]:
    vals = amt_probe.filter(pl.col("Is Laundering") == lab)["log1p_amount"].to_numpy()
    ax.hist(vals, bins=60, alpha=0.55, color=color, label=name, density=True)
ax.set_title("Log1p(Amount Paid) — same-currency probe")
ax.set_xlabel("log1p(amount)")
ax.set_ylabel("density")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- FX / cross-currency, cross-bank, self-transfer ---
fx = collect_scan(
    lf_raw.group_by(
        (pl.col("Payment Currency") != pl.col("Receiving Currency")).alias("is_cross_currency")
    ).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("is_cross_currency")
)
print("Cross-currency vs same-currency:")
print(fx.with_columns((pl.col("rate") / full_rate).alias("lift")))

bank_dir = collect_scan(
    lf_raw.group_by((pl.col("From Bank") != pl.col("To Bank")).alias("is_cross_bank")).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").sum().alias("n_laundering"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("is_cross_bank")
)
print("\nIntra-bank vs cross-bank:")
print(bank_dir.with_columns((pl.col("rate") / full_rate).alias("lift")))

self_stats = collect_scan(
    lf_raw.group_by(
        (
            (pl.col("Account") == pl.col("Account.1"))
            & (pl.col("From Bank") == pl.col("To Bank"))
        ).alias("is_self_transfer")
    ).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("is_self_transfer")
)
print("\nSelf-transfer:")
print(self_stats.with_columns((pl.col("rate") / full_rate).alias("lift")))


In [ ]:
# --- time patterns: hour-of-day, weekday, daily rate ---
time_df = collect_scan(
    lf_ts.group_by(pl.col("ts").dt.hour().alias("hour")).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("hour")
)
dow = collect_scan(
    lf_ts.group_by(pl.col("ts").dt.weekday().alias("dow")).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("dow")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time_df["hour"], time_df["rate"] * 100, marker="o", color="crimson")
axes[0].axhline(full_rate * 100, linestyle="--", color="steelblue")
axes[0].set_title("Laundering rate by hour of day")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("rate (%)")

dow_names = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
axes[1].bar(
    [dow_names.get(int(d), str(d)) for d in dow["dow"].to_list()],
    (dow["rate"] * 100).to_list(),
    color="crimson",
)
axes[1].axhline(full_rate * 100, linestyle="--", color="steelblue")
axes[1].set_title("Laundering rate by weekday")
axes[1].set_ylabel("rate (%)")
plt.tight_layout()
plt.show()

daily = collect_scan(
    lf_ts.group_by(pl.col("ts").dt.date().alias("day")).agg(
        pl.len().alias("n"),
        pl.col("Is Laundering").sum().alias("n_laundering"),
        pl.col("Is Laundering").mean().alias("rate"),
    ).sort("day")
)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily["day"].to_list(), (daily["rate"] * 100).to_list(), color="crimson", linewidth=1)
ax.axvline(PRIMARY_END.date(), color="gray", linestyle="--", label="primary window end")
ax.set_title("Daily laundering rate (post-window tail should jump — we drop it)")
ax.set_ylabel("rate (%)")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Cleaning + ≥1M neighborhood subsample

Attach GNN fields, then build the **1M / ~4% / 60–20–20** train-fraud-seeded subsample (same recipe as `EDA-IBM.ipynb`).


In [ ]:
def dest_account_col(names):
    """Polars may keep CSV 'Account.1' or rename the 2nd Account column to Account_duplicated_0."""
    for name in ("Account.1", "Account_duplicated_0", "Account_right"):
        if name in names:
            return name
    raise KeyError(
        "Could not find destination account column. Schema names: " + ", ".join(names)
    )


def with_graph_fields(lazy: pl.LazyFrame) -> pl.LazyFrame:
    """Composite node IDs, timestamps, AUD amounts, and AML flags."""
    names = list(lazy.collect_schema().names())
    dest_col = dest_account_col(names)
    print(f"Destination account column: {dest_col!r}")
    return (
        lazy.with_columns(
            pl.col("From Bank").cast(pl.Utf8),
            pl.col("To Bank").cast(pl.Utf8),
            pl.col("Account").cast(pl.Utf8),
            pl.col(dest_col).cast(pl.Utf8).alias("_dest_acct"),
        )
        .with_columns(
            (pl.col("Account") + "_" + pl.col("From Bank")).alias("sender_id"),
            (pl.col("_dest_acct") + "_" + pl.col("To Bank")).alias("receiver_id"),
            pl.col("Timestamp").str.to_datetime("%Y/%m/%d %H:%M", strict=False).alias("ts"),
            (
                pl.col("Amount Paid")
                * pl.col("Payment Currency").replace_strict(RATES, default=1.0)
            ).alias("amount_paid_aud"),
            (
                pl.col("Amount Received")
                * pl.col("Receiving Currency").replace_strict(RATES, default=1.0)
            ).alias("amount_received_aud"),
        )
        .with_columns(
            ((pl.col("amount_paid_aud") + pl.col("amount_received_aud")) / 2).alias("amount"),
            (pl.col("sender_id") == pl.col("receiver_id")).cast(pl.Int8).alias("is_self_transfer"),
            (pl.col("From Bank") != pl.col("To Bank")).cast(pl.Int8).alias("is_cross_bank"),
            (pl.col("Payment Currency") != pl.col("Receiving Currency")).cast(pl.Int8).alias("is_cross_currency"),
        )
    )


def accounts_frame(df: pl.DataFrame) -> pl.DataFrame:
    return pl.concat(
        [
            df.select(pl.col("sender_id").alias("account")),
            df.select(pl.col("receiver_id").alias("account")),
        ]
    ).unique()


def expand_account_pool(lazy_all: pl.LazyFrame, seed_accounts: pl.DataFrame, max_hops: int):
    """Grow the account set hop by hop via joins (no giant Python is_in lists)."""
    pool = seed_accounts.unique()
    hop_sizes = [pool.height]
    print(f"0-hop (train laundering) accounts: {pool.height:,}")
    nonfraud = lazy_all.filter(pl.col("Is Laundering") == 0)
    used_hops = 0
    hop_bar = tqdm(range(1, max_hops), desc="Hop expansion", unit="hop", leave=True)
    for hop in hop_bar:
        hop_bar.set_postfix_str(f"pool={pool.height:,}")
        pool_l = pool.lazy()
        via_src = (
            nonfraud.join(pool_l, left_on="sender_id", right_on="account", how="semi")
            .select(pl.col("receiver_id").alias("account"))
        )
        via_dst = (
            nonfraud.join(pool_l, left_on="receiver_id", right_on="account", how="semi")
            .select(pl.col("sender_id").alias("account"))
        )
        new_accts = collect_scan(
            pl.concat([via_src, via_dst]).unique(),
            desc=f"hop {hop} account expansion (train-window join)",
        )
        pool = pl.concat([pool, new_accts]).unique()
        hop_sizes.append(pool.height)
        used_hops = hop
        hop_bar.set_postfix_str(f"pool={pool.height:,}")
        print(f"Hop {hop} pool size: {pool.height:,} unique accounts")
        if pool.height >= MAX_POOL_ACCOUNTS:
            print("Account-pool cap hit — stopping expansion (4 hops would swallow HI-Medium).")
            break
    hop_bar.close()
    return pool, hop_sizes, used_hops


def hash_sample(lazy: pl.LazyFrame, n: int, seed: int) -> pl.DataFrame:
    """Approximate sample of n rows from a huge lazy frame without collecting all of it."""
    if n <= 0:
        return collect_scan(lazy.head(0))
    n_avail = int(collect_scan(lazy.select(pl.len()), desc="count rows before hash-sample").item())
    if n_avail == 0:
        return collect_scan(lazy.head(0), desc="empty hash-sample")
    if n_avail <= n:
        return collect_scan(lazy, desc=f"collect all {n_avail:,} hash-sample candidates")
    keep = min(1_000_000, max(1, int(1_000_000 * (n * 1.4) / n_avail) + 1))
    over = collect_scan(
        lazy.with_columns((pl.col("sender_id").hash(seed=seed) % 1_000_000).alias("_b"))
        .filter(pl.col("_b") < keep)
        .drop("_b")
    )
    if over.height >= n:
        return over.sample(n=n, seed=seed)
    extra = collect_scan(
        lazy.with_columns((pl.col("receiver_id").hash(seed=seed + 1) % 1_000_000).alias("_b"))
        .filter(pl.col("_b") < keep)
        .drop("_b")
    )
    merged = pl.concat([over, extra]).unique()
    if merged.height >= n:
        return merged.sample(n=n, seed=seed)
    return merged



def primary_window_cuts(lazy: pl.LazyFrame):
    """0.6 / 0.8 quantiles of ts on the (already window-filtered) scan."""
    cuts = collect_scan(
        lazy.select(
            pl.col("ts").dt.timestamp("us").quantile(0.6).alias("t_train_us"),
            pl.col("ts").dt.timestamp("us").quantile(0.8).alias("t_val_us"),
        ),
        desc="primary-window ts quantiles 0.6 / 0.8",
    )
    t_train_end = (
        pl.Series("t", [int(cuts["t_train_us"].item())], dtype=pl.Int64)
        .cast(pl.Datetime("us"))
        .item()
    )
    t_val_end = (
        pl.Series("t", [int(cuts["t_val_us"].item())], dtype=pl.Int64)
        .cast(pl.Datetime("us"))
        .item()
    )
    return t_train_end, t_val_end


def sample_cache_is_current(df: pl.DataFrame) -> bool:
    need = {"seed_policy", "t_train_end", "t_val_end", "sample_source"}
    if not need.issubset(df.columns):
        return False
    if str(df["seed_policy"][0]) != SEED_POLICY:
        return False
    rate = float(df["Is Laundering"].mean())
    if df.height < MIN_ROWS:
        return False
    return abs(rate - MIN_FRAUD_RATE) < 0.012


def sample_gnn_neighborhood(lazy: pl.LazyFrame):
    """Train-fraud seeds, hops on train-time non-fraud, ≥1M rows (~4% mix)."""
    print("=== Subsample formula (train-fraud seeds, ≥1M rows, ~4% mix) ===")
    print(f"MIN_ROWS        = {MIN_ROWS:,}")
    print(f"MIN_FRAUD_RATE  = {MIN_FRAUD_RATE:.2%}")
    print()

    t_train_end, t_val_end = primary_window_cuts(lazy)
    print(f"t_train_end (q=0.6) = {t_train_end}")
    print(f"t_val_end   (q=0.8) = {t_val_end}")
    print()

    lf_train = lazy.filter(pl.col("ts") <= pl.lit(t_train_end))
    fraud_train = collect_scan(
        lf_train.filter(pl.col("Is Laundering") == 1),
        desc="collect TRAIN-period laundering rows",
    )
    print(f"train-period laundering (seeds): {fraud_train.height:,}")

    seed_accts = accounts_frame(fraud_train)
    pool, hop_sizes, used_hops = expand_account_pool(lf_train, seed_accts, MAX_HOPS)
    accts = pool.get_column("account")
    in_pool = pl.col("sender_id").is_in(accts) | pl.col("receiver_id").is_in(accts)

    fraud_keep = collect_scan(
        lazy.filter((pl.col("Is Laundering") == 1) & in_pool),
        desc="collect pool-account laundering (train + later)",
    )
    n_fraud = fraud_keep.height
    n_total_target = int(round(n_fraud / MIN_FRAUD_RATE))
    n_total = max(MIN_ROWS, n_total_target)
    if n_fraud / n_total < MIN_FRAUD_RATE:
        print(
            f"NOTE: max({MIN_ROWS:,}, {n_total_target:,}) = {n_total:,} is "
            f"{n_fraud / n_total:.2%} laundering (< {MIN_FRAUD_RATE:.0%} target). "
            f"Keeping {n_total:,} rows — 1M-row pipeline prefers the row floor."
        )
    n_legit = n_total - n_fraud
    print(f"n_fraud (pool accounts) = {n_fraud:,}")
    print(f"n_total                 = {n_total:,}")
    print(f"n_legit                 = {n_legit:,}")
    print(f"target subsample rate   = {n_fraud / n_total:.4%}")
    print()

    nonfraud = lazy.filter(pl.col("Is Laundering") == 0)
    neigh_lf = nonfraud.filter(in_pool)
    rest_lf = nonfraud.filter(~in_pool)

    n_neigh_avail = int(collect_scan(neigh_lf.select(pl.len()), desc="count neighborhood legit").item())
    print(f"Neighborhood legit available: {n_neigh_avail:,}")
    n_neigh_keep = min(n_neigh_avail, n_legit)
    n_rest_keep = n_legit - n_neigh_keep
    print(f"Keeping neighborhood legit:   {n_neigh_keep:,}")
    print(f"Filling random legit:         {n_rest_keep:,}")

    neigh_s = hash_sample(neigh_lf, n_neigh_keep, RANDOM_STATE).with_columns(
        pl.lit("neighborhood").alias("sample_source")
    )
    rest_s = hash_sample(rest_lf, n_rest_keep, RANDOM_STATE + 7).with_columns(
        pl.lit("random").alias("sample_source")
    )
    fraud_s = fraud_keep.with_columns(pl.lit("laundering").alias("sample_source"))

    sampled = pl.concat([fraud_s, neigh_s, rest_s], how="diagonal_relaxed").sample(
        fraction=1.0, seed=RANDOM_STATE
    )
    sampled = sampled.with_columns(
        pl.lit(SEED_POLICY).alias("seed_policy"),
        pl.lit(t_train_end).alias("t_train_end"),
        pl.lit(t_val_end).alias("t_val_end"),
    )
    rate = sampled["Is Laundering"].mean()
    print()
    print(f"Final subsample: {sampled.height:,} rows")
    print(f"  laundering: {int(sampled['Is Laundering'].sum()):,}")
    print(f"  legit:      {int((sampled['Is Laundering'] == 0).sum()):,}")
    print(f"  rate:       {rate:.4%}")
    assert sampled.height >= MIN_ROWS, (
        f"subsample {sampled.height:,} < MIN_ROWS {MIN_ROWS:,}"
    )
    # MIN_ROWS can force mix below MIN_FRAUD_RATE (e.g. ~30k fraud / 1M ≈ 3%).
    # That is intentional: prefer the 1M floor over shrinking to hit 4%.
    if rate < MIN_FRAUD_RATE:
        print(
            f"NOTE: mix {rate:.2%} < target {MIN_FRAUD_RATE:.0%} because "
            f"MIN_ROWS={MIN_ROWS:,} with n_fraud={int(sampled['Is Laundering'].sum()):,}."
        )
    assert rate >= (n_fraud / MIN_ROWS) - 1e-9, (
        f"mix {rate:.4%} is below the 1M-floor rate floor "
        f"({n_fraud / MIN_ROWS:.4%}); fraud rows were dropped."
    )
    return sampled, neigh_s.height, rest_s.height, hop_sizes, used_hops, t_train_end, t_val_end



# --- cleaning + primary-window subsample ---
lf = with_graph_fields(lf_raw)
print("Graph-field schema:")
print(lf.collect_schema())
print("\nPreview with IDs + flags:")
display(
    collect_scan(
        lf.select(
            "ts", "sender_id", "receiver_id", "amount",
            "Payment Format", "is_self_transfer", "is_cross_bank", "is_cross_currency",
            "Is Laundering",
        ).head(5)
    )
)

lf_primary = lf.filter(
    (pl.col("ts") >= pl.lit(PRIMARY_START)) & (pl.col("ts") <= pl.lit(PRIMARY_END))
)

df_sampled = None
if SAMPLE_PATH.exists():
    cached = pl.read_parquet(SAMPLE_PATH)
    if sample_cache_is_current(cached):
        print(f"Found {SAMPLE_PATH.resolve()} — loading cached subsample ({SEED_POLICY}, ≥1M, ~4%).")
        df_sampled = cached
        t_train_end = df_sampled["t_train_end"][0]
        t_val_end = df_sampled["t_val_end"][0]
        n_neigh_s = df_sampled.filter(pl.col("sample_source") == "neighborhood").height
        n_rest_s = df_sampled.filter(pl.col("sample_source") == "random").height
        hop_sizes, used_hops = [], None
        print(df_sampled.shape, "rate", f"{df_sampled['Is Laundering'].mean():.4%}")
        print(f"t_train_end = {t_train_end} | t_val_end = {t_val_end}")
    else:
        print(f"Stale {SAMPLE_PATH.name} (need seed_policy={SEED_POLICY!r} and ≥1M / ~4% mix) — rebuilding.")

if df_sampled is None:
    df_sampled, n_neigh_s, n_rest_s, hop_sizes, used_hops, t_train_end, t_val_end = (
        sample_gnn_neighborhood(lf_primary)
    )
    df_sampled.write_parquet(SAMPLE_PATH)
    print(f"\nWrote {SAMPLE_PATH.resolve()}")

n_fraud_s = int(df_sampled["Is Laundering"].sum())
t0 = df_sampled["ts"].min()
df_sampled = df_sampled.with_columns(
    ((pl.col("ts") - t0).dt.total_seconds() / 3600.0).alias("hours_from_start")
)
rate_s = float(df_sampled["Is Laundering"].mean())
print(f"\nASSERTS: n={df_sampled.height:,} (>= {MIN_ROWS:,})  rate={rate_s:.4%} (target ~{MIN_FRAUD_RATE:.0%}; 1M floor may land ~3%)")
print(f"time cuts: train <= {t_train_end} | val <= {t_val_end}")

# --- composition bar: fraud + neighborhood + random ---
if "sample_source" in df_sampled.columns:
    n_neigh_s = df_sampled.filter(pl.col("sample_source") == "neighborhood").height
    n_rest_s = df_sampled.filter(pl.col("sample_source") == "random").height
composition_counts = [n_fraud_s, n_neigh_s, n_rest_s]
composition_labels = [
    "Laundering\n(train-pool)",
    "k-hop legit\n(neighborhood)",
    "Random legit\n(background)",
]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(composition_labels, composition_counts, color=["crimson", "orange", "steelblue"])
ax.set_title(f"Subsample composition ({df_sampled.height:,} rows, {rate_s*100:.2f}% laundering)")
ax.set_ylabel("transactions (edges)")
for bar, val in zip(bars, composition_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{int(val):,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

print(
    f"Formula recap: n_total = max({MIN_ROWS:,}, round({n_fraud_s:,} / {MIN_FRAUD_RATE})) "
    f"= {max(MIN_ROWS, int(round(n_fraud_s / MIN_FRAUD_RATE))):,}; "
    f"emitted {df_sampled.height:,} rows at {rate_s:.2%}"
)
if hop_sizes:
    print("Hop pool sizes:", hop_sizes)

print("""
GNN design notes
================
Task:        EDGE classification (Is Laundering on each transaction).
Graph:       directed MULTIGRAPH, accounts = nodes, transactions = edges.
Self-loops:  KEEP. Banks: do NOT one-hot — use is_cross_bank.
Node feats:  log1p(in_degree), log1p(out_degree) from TRAIN edges only.
Edge feats:  RobustScaler(amount, hours_from_start) train-fit, clip ±10
             Payment Format one-hot (drop_first, train levels)
             is_cross_bank, is_cross_currency, is_self_transfer
Split:       calendar 60/20/20 on Sep 1 – Sep 16; split-prefix message passing.
Metric:      unleaked AUPRC. Target band 0.70–0.80; >0.82 triggers a leak audit.
""")
print("Payment Format levels:")
print(df_sampled["Payment Format"].value_counts().sort("count", descending=True))


df = df_sampled.to_pandas()
# pandas path expects Timestamp column for the graph cell
if "Timestamp" not in df.columns and "ts" in df.columns:
    df["Timestamp"] = pd.to_datetime(df["ts"])
print(f"pandas df: {len(df):,} rows | columns={list(df.columns)}")
display(df.head())


# Data Preprocessing

In [ ]:
# FX + AML flags: skip if the Polars subsample already attached them.
if "amount" not in df.columns:
    print("Using fixed FX rates (no live API)")
    print(RATES)
    print("--------------------------------")
    print("Currency before conversion")
    print("--------------------------------")
    print(df[["Amount Paid", "Payment Currency", "Amount Received", "Receiving Currency"]].head())

    df["Amount Paid"] = df["Amount Paid"] * df["Payment Currency"].map(RATES)
    df["Amount Received"] = df["Amount Received"] * df["Receiving Currency"].map(RATES)
    print("--------------------------------")
    print("Currency After conversion")
    print("--------------------------------")
    df["amount"] = (df["Amount Paid"] + df["Amount Received"]) / 2
    print(df[["Amount Paid", "Payment Currency", "Amount Received", "Receiving Currency", "amount"]].head())
    print(df["amount"].isna().sum())
else:
    print("Polars subsample already has amount / flags — skipping FX conversion.")
    print(df[["amount", "sender_id", "receiver_id"]].head())

data = df.copy()
if "ts" in data.columns:
    data["Timestamp"] = pd.to_datetime(data["ts"], errors="coerce")
else:
    data["Timestamp"] = pd.to_datetime(data["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")

if "is_self_transfer" not in data.columns:
    data["is_self_transfer"] = (data["sender_id"] == data["receiver_id"]).astype(float)
else:
    data["is_self_transfer"] = data["is_self_transfer"].astype(float)
if "is_cross_bank" not in data.columns:
    data["is_cross_bank"] = (data["From Bank"] != data["To Bank"]).astype(float)
else:
    data["is_cross_bank"] = data["is_cross_bank"].astype(float)
if "is_cross_currency" not in data.columns:
    data["is_cross_currency"] = (
        data["Payment Currency"] != data["Receiving Currency"]
    ).astype(float)
else:
    data["is_cross_currency"] = data["is_cross_currency"].astype(float)


# Converting to Graph

Same recipe as `750k_row_trial.ipynb` / unleaked classical GNN: primary window, **timestamp cuts** from the subsample (`t_train_end` / `t_val_end`), train-only scaler / one-hot, **sorted** account ids, **2-d log1p in/out degree** on train edges only (no currency avgs).

Split-prefix message passing (wired in `message_graph`, not the QGNN class): train MP = train edges; val MP = train edges; test MP = train+val. FANOUTS still caps the local neighbourhood at forward time, but only inside that pre-sliced graph.


In [ ]:
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from torch_geometric.utils import degree

GRAPH_CACHE_PATH = Path("ibm_graph_cache.pt")
SERVING_PATH = Path("ibm_graph.pt")

bundle = None
if GRAPH_CACHE_PATH.is_file():
    bundle = torch.load(GRAPH_CACHE_PATH, weights_only=False)
    nf = bundle.get("node_features")
    stale = (
        nf is None
        or nf.ndim != 2
        or nf.shape[1] != 2  # log1p in/out degree only
        or "val_edge_index" not in bundle
        or "train_edge_index" not in bundle
        or bundle.get("mp_policy") != "split_prefix_v1"
        or bundle.get("node_feat_policy") != "train_log1p_degree_v1"
        or bundle.get("scaler") is None
        or bundle.get("cat_encoder") is None
        or bundle.get("t0") is None
    )
    if stale:
        print(
            "Stale ibm_graph_cache.pt (need split_prefix_v1 MP, 2-d train log1p degrees, scaler/encoder/t0) — rebuilding"
        )
        bundle = None

if bundle is not None:
    edge_index = bundle["edge_index"]
    edge_attr = bundle["edge_attr"]
    edge_labels = bundle["edge_labels"]
    train_mask = bundle["train_mask"]
    val_mask = bundle["val_mask"]
    test_mask = bundle["test_mask"]
    node_features = bundle["node_features"]
    node_mapping = bundle["node_mapping"]
    pos_weight_train = bundle["pos_weight_train"]
    train_edge_index = bundle["train_edge_index"]
    train_edge_attr = bundle["train_edge_attr"]
    val_edge_index = bundle["val_edge_index"]
    val_edge_attr = bundle["val_edge_attr"]
    scaler = bundle["scaler"]
    cat_encoder = bundle["cat_encoder"]
    t0 = bundle["t0"]
    clip = float(bundle.get("clip", 10.0))
    feature_names = list(bundle.get("feature_names", []))
    CONT_COLS = list(bundle.get("cont_cols") or ["amount", "hours_from_start"])
    CAT_COLS = list(bundle.get("cat_cols") or ["Payment Format"])
    BIN_COLS = list(bundle.get("bin_cols") or ["is_self_transfer", "is_cross_bank", "is_cross_currency"])
    print(f"Loaded cached graph tensors from {GRAPH_CACHE_PATH.resolve()}")
    print(
        f"nodes={len(node_mapping):,} | edges={edge_index.size(1):,} | "
        f"node_dim={node_features.shape[1]} | edge_attr_dim={edge_attr.shape[1]}"
    )
    print(
        f"pre-sliced MP: train_edges={train_edge_index.size(1):,} | "
        f"val_subgraph={val_edge_index.size(1):,}"
    )
    print(f"pos_weight (train): {float(pos_weight_train):.2f}")
else:
    # HI-Medium primary window (Kaggle). Rows after this are ~100% laundering — do not use them in the split.
    PRIMARY_START = datetime(2022, 9, 1)
    PRIMARY_END = datetime(2022, 9, 16, 23, 59)

    n_before = len(data)
    post = data.loc[data["Timestamp"] > PRIMARY_END]
    print(
        f"Post-window rows dropped: {len(post):,} | "
        f"laundering rate={post['Is Laundering'].mean() * 100 if len(post) else float('nan'):.2f}%"
    )
    data = data.loc[
        (data["Timestamp"] >= PRIMARY_START) & (data["Timestamp"] <= PRIMARY_END)
    ].copy()
    print(f"Primary-window rows: {len(data):,} (was {n_before:,})")
    if data.empty:
        raise ValueError("No rows left after applying the 1–16 Sep 2022 primary window.")

    # ---- timestamp split (same cuts as the EDA-IBM subsample)
    data = data.sort_values("Timestamp").reset_index(drop=True)
    t0 = data["Timestamp"].min()
    data["hours_from_start"] = (data["Timestamp"] - t0).dt.total_seconds() / 3600.0
    y = data["Is Laundering"].astype(int).to_numpy()

    n = len(data)
    if "t_train_end" not in data.columns or "t_val_end" not in data.columns:
        raise ValueError(
            "subsample missing t_train_end / t_val_end — rebuild parquet with seed_policy=train_fraud_v1"
        )
    t_train_end = pd.Timestamp(data["t_train_end"].iloc[0])
    t_val_end = pd.Timestamp(data["t_val_end"].iloc[0])
    ts = pd.to_datetime(data["Timestamp"])
    idx_train = np.flatnonzero((ts <= t_train_end).to_numpy())
    idx_val = np.flatnonzero(((ts > t_train_end) & (ts <= t_val_end)).to_numpy())
    idx_test = np.flatnonzero((ts > t_val_end).to_numpy())
    if idx_train.size == 0 or idx_val.size == 0 or idx_test.size == 0:
        raise ValueError(
            f"Empty split: train={idx_train.size} val={idx_val.size} test={idx_test.size}"
        )

    print("train ends:", t_train_end)
    print("val ends:  ", t_val_end)
    print("test ends: ", data["Timestamp"].max())
    print(
        f"Primary-window edges: {n:,} | train {idx_train.size:,} | "
        f"val {idx_val.size:,} | test {idx_test.size:,}"
    )

    for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
        rate = data.loc[idx, "Is Laundering"].mean() * 100
        print(f"{name}: fraud rate = {rate:.2f}%")

    # --- Edge features (same recipe as EDA-IBM): train-only scaler / one-hot ---
    CONT_COLS = ["amount", "hours_from_start"]
    CAT_COLS = ["Payment Format"]
    BIN_COLS = ["is_self_transfer", "is_cross_bank", "is_cross_currency"]

    X_cont = data[CONT_COLS].astype(float).to_numpy()
    X_cont_scaled = X_cont.copy()
    scaler = RobustScaler()
    X_cont_scaled[idx_train] = scaler.fit_transform(X_cont[idx_train])
    X_cont_scaled[idx_val] = scaler.transform(X_cont[idx_val])
    X_cont_scaled[idx_test] = scaler.transform(X_cont[idx_test])
    X_cont_scaled = np.clip(X_cont_scaled, -10, 10)

    try:
        cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)
    X_cat_raw = data[CAT_COLS].astype(str).to_numpy()
    cat_encoder.fit(X_cat_raw[idx_train])
    X_cat = cat_encoder.transform(X_cat_raw).astype(np.float32)
    cat_names = list(cat_encoder.get_feature_names_out(CAT_COLS))

    X_bin = data[BIN_COLS].astype(float).to_numpy()
    edge_features = np.hstack([X_cont_scaled.astype(np.float32), X_cat, X_bin])
    feature_names = CONT_COLS + cat_names + BIN_COLS
    print("edge feature_names:", feature_names)

    # Sorted unique account ids (same as EDA-IBM np.unique), not pandas first-seen order
    all_nodes = np.unique(
        np.concatenate([data["sender_id"].to_numpy(), data["receiver_id"].to_numpy()])
    )
    node_mapping = {name: i for i, name in enumerate(all_nodes)}
    src = data["sender_id"].map(node_mapping).to_numpy()
    dst = data["receiver_id"].map(node_mapping).to_numpy()
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    edge_labels = torch.tensor(y, dtype=torch.float)

    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)
    train_mask[idx_train] = True
    val_mask[idx_val] = True
    test_mask[idx_test] = True

    num_nodes_graph = len(node_mapping)
    pos_weight_train = float((y[idx_train] == 0).sum() / max((y[idx_train] == 1).sum(), 1))

    # Node features (same as 750k_row_trial): train-only log1p in/out degree
    train_edge_index = edge_index[:, train_mask]
    out_deg = torch.log1p(degree(train_edge_index[0], num_nodes=num_nodes_graph, dtype=torch.float))
    in_deg = torch.log1p(degree(train_edge_index[1], num_nodes=num_nodes_graph, dtype=torch.float))
    node_features = torch.stack([in_deg, out_deg], dim=1)

    # Split-prefix: val_edge_index is train+val (used as TEST MP). Val eval uses train_edge_index.
    train_edge_attr = edge_attr[train_mask]
    val_subgraph_mask = train_mask | val_mask
    val_edge_index = edge_index[:, val_subgraph_mask]
    val_edge_attr = edge_attr[val_subgraph_mask]

    print(
        f"nodes={len(node_mapping):,} | edges={n:,} | "
        f"node_dim={node_features.shape[1]} | edge_attr_dim={edge_attr.shape[1]}"
    )
    print(
        f"pre-sliced MP: train_edges={train_edge_index.size(1):,} | "
        f"val_subgraph={val_edge_index.size(1):,}"
    )
    print(f"pos_weight (train): {pos_weight_train:.2f}")
    for name, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
        yy = y[m.numpy()]
        print(f"{name:5s} | n={len(yy):6d} | fraud={yy.sum():5d} | rate={yy.mean()*100:.3f}%")
    assert edge_index.shape[1] == edge_attr.shape[0] == edge_labels.shape[0] == n
    assert node_features.shape[0] == num_nodes_graph and node_features.shape[1] == 2

    torch.save(
        {
            "edge_index": edge_index,
            "edge_attr": edge_attr,
            "edge_labels": edge_labels,
            "train_mask": train_mask,
            "val_mask": val_mask,
            "test_mask": test_mask,
            "node_features": node_features,
            "num_nodes": len(node_mapping),
            "node_mapping": node_mapping,
            "pos_weight_train": float(pos_weight_train),
            "train_edge_index": train_edge_index,
            "train_edge_attr": train_edge_attr,
            "val_edge_index": val_edge_index,
            "val_edge_attr": val_edge_attr,
            "feature_names": feature_names,
            "mp_policy": "split_prefix_v1",
            "node_feat_policy": "train_log1p_degree_v1",
            "t_train_end": str(t_train_end),
            "t_val_end": str(t_val_end),
            "scaler": scaler,
            "cat_encoder": cat_encoder,
            "t0": t0.isoformat() if hasattr(t0, "isoformat") else str(t0),
            "clip": 10.0,
            "cont_cols": list(CONT_COLS),
            "cat_cols": list(CAT_COLS),
            "bin_cols": list(BIN_COLS),
            "rates": dict(RATES),
        },
        GRAPH_CACHE_PATH,
    )
    print(f"Saved graph tensors to {GRAPH_CACHE_PATH}")


# Train model

## Model Definition

In [ ]:
import json
from pathlib import Path

import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import time
import os
import wandb
from torch_geometric.utils import softmax

n_qubits = 2
# Same values as best_params.json — used if the file is not found.
_FALLBACK_BEST_PARAMS = {
    "optimizer": "AdamW",
    "lr": 0.009378125543991336,
    "weight_decay": 7.476801079666406e-05,
    "hidden_dim": 64,
    "dropout": 0.11156817334385223,
    "pos_weight_factor": 0.8048234311904515,
}


def _find_best_params():
    candidates = [
        Path("best_params.json"),
        Path("/kaggle/working/best_params.json"),
        Path("/kaggle/input/qgnn-best-params/best_params.json"),
        Path("/kaggle/input/lachlanbruce/qgnn-best-params/best_params.json"),
        Path("/kaggle/input/datasets/lachlanbruce/qgnn-best-params/best_params.json"),
    ]
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for pattern in ("*/best_params.json", "*/*/best_params.json", "*/*/*/best_params.json"):
            candidates.extend(kaggle_in.glob(pattern))
    seen, uniq = set(), []
    for p in candidates:
        key = str(p)
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return next((p for p in uniq if p.is_file()), None), uniq


BEST_PARAMS_PATH, _ = _find_best_params()


def make_qdevice(n_qubits):
    if torch.cuda.is_available():
        try:
            return qml.device("lightning.gpu", wires=n_qubits)
        except Exception as err:
            print("lightning.gpu unavailable:", err)
    try:
        return qml.device("lightning.qubit", wires=n_qubits)
    except Exception as err:
        print("lightning.qubit unavailable:", err)
        return qml.device("default.qubit", wires=n_qubits)


dev = make_qdevice(n_qubits)
print("Set up device")
print("PennyLane device:", dev.name)

@qml.qnode(dev, interface="torch", diff_method="adjoint")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(
            quantum_circuit, {"weights": (1, n_qubits, 3)}  # depth 1 for speed
        )

    def forward(self, x):
        # TorchLayer can return CPU even when x is CUDA — keep the VQC on x's device.
        return self.qlayer(x).to(x.device)


FANOUTS = (2,)  # one hop, sampled inside the pre-sliced train/val/full graph
MAX_Q_EDGES = 128  # hard cap on VQC edges per forward (speed)
def build_incident_csr(edge_index, num_nodes):
    """CSR of incident edge ids (src or dst) for O(degree) neighbor lookup."""
    src, dst = edge_index[0], edge_index[1]
    eids = torch.arange(edge_index.size(1), device=edge_index.device)
    nodes = torch.cat([src, dst])
    edges = torch.cat([eids, eids])
    order = torch.argsort(nodes)
    nodes = nodes[order]
    edges = edges[order]
    counts = torch.bincount(nodes, minlength=num_nodes)
    ptr = torch.empty(num_nodes + 1, dtype=torch.long, device=edge_index.device)
    ptr[0] = 0
    ptr[1:] = torch.cumsum(counts, dim=0)
    return ptr, edges


def gather_incident_eids(node_set, ptr, eids):
    if node_set.numel() == 0:
        return eids.new_empty(0)
    starts = ptr[node_set]
    deg = ptr[node_set + 1] - starts
    nz = deg > 0
    if not torch.any(nz):
        return eids.new_empty(0)
    starts = starts[nz]
    deg = deg[nz]
    total = int(deg.sum().item())
    row = torch.repeat_interleave(torch.arange(starts.numel(), device=node_set.device), deg)
    inner = torch.arange(total, device=node_set.device) - torch.repeat_interleave(
        torch.cumsum(deg, 0) - deg, deg
    )
    return eids[starts[row] + inner].unique()


def sample_subgraph(seed_nodes, edge_index, edge_attr, fanouts, ptr, eids):
    """k-hop sub-sector around the batch, with a cap per hop.

    `ptr` / `eids` must match `edge_index` (train, train+val, or full).
    Each hop keeps at most fanout * |current nodes| incident edges, then
    expands the node set. Not strict per-node neighbour sampling, but it
    stops the IBM graph exploding into the full training set.
    """
    node_set = seed_nodes.unique()
    keep_parts = []
    for fanout in fanouts:
        inc = gather_incident_eids(node_set, ptr, eids)
        if inc.numel() == 0:
            break
        cap = fanout * int(node_set.numel())
        if inc.numel() > cap:
            pick = torch.randperm(inc.numel(), device=inc.device)[:cap]
            inc = inc[pick]
        keep_parts.append(inc)
        node_set = torch.cat([node_set, edge_index[0, inc], edge_index[1, inc]]).unique()
    if not keep_parts:
        return edge_index[:, :0], edge_attr[:0]
    keep = torch.cat(keep_parts).unique()
    return edge_index[:, keep], edge_attr[keep]


def compact_nodes(sub_index, extra_nodes):
    used = torch.cat([extra_nodes, sub_index.reshape(-1)]).unique()
    mapper = torch.empty(int(used.max().item()) + 1, dtype=torch.long, device=used.device)
    mapper[used] = torch.arange(used.numel(), device=used.device)
    sub_local = mapper[sub_index] if sub_index.numel() else sub_index
    return used, mapper, sub_local


class QuantumWeightedConv(nn.Module):
    """One hop: reuse precomputed α_ij -> weighted aggregate + classical update."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.update = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, x, edge_index, alpha):
        if edge_index.size(1) == 0:
            return x

        src, dst = edge_index[0], edge_index[1]
        msg = x[src] * alpha.unsqueeze(-1)
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, msg)

        return F.relu(self.update(torch.cat([x, agg], dim=-1)))


class QGNN(nn.Module):
    def __init__(self, num_nodes, node_attr_dim, edge_attr_dim, hidden_dim, fanouts=FANOUTS, dropout=0.3):
        super().__init__()
        self.fanouts = fanouts
        self.dropout = dropout
        self.node_enc = nn.Linear(node_attr_dim, hidden_dim)
        self.q_layer = QuantumLayer()  # single shared VQC
        # quantum attention computed once, then reused across hops
        self.to_qubits = nn.Linear(hidden_dim * 2 + edge_attr_dim, n_qubits)
        self.att = nn.Linear(n_qubits, 1)
        self.gat_att = nn.Linear(hidden_dim * 2 + edge_attr_dim, 1)
        self.convs = nn.ModuleList(
            [QuantumWeightedConv(hidden_dim) for _ in fanouts]
        )
        # classical head: hops already mixed quantum weights into h_src / h_dst
        self.classifier = nn.Linear(hidden_dim * 2 + edge_attr_dim, 1)

    def quantum_alphas(self, x, edge_index, edge_attr):
        """One VQC pass over subgraph edges -> softmax α_ij (brief §3)."""
        if edge_index.size(1) == 0:
            return edge_index.new_zeros(0)
        src, dst = edge_index[0], edge_index[1]
        pair = torch.cat([x[src], x[dst], edge_attr], dim=-1)
        logits = F.leaky_relu(self.gat_att(pair).squeeze(-1), 0.2)

        angles = math.pi * torch.tanh(
            self.to_qubits(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        )
        n = angles.size(0)
        if n > MAX_Q_EDGES:
            keep = torch.randperm(n, device=angles.device)[:MAX_Q_EDGES]
            q_logits = F.leaky_relu(self.att(self.q_layer(angles[keep])).squeeze(-1), 0.2)
            logits = logits.clone()
            logits[keep] = logits[keep] + q_logits
        else:
            q_logits = F.leaky_relu(self.att(self.q_layer(angles)).squeeze(-1), 0.2)
            logits = logits + q_logits
        return softmax(logits, dst)

    def forward(self, x, mp_edge_index, mp_edge_attr, edge_index, edge_attr, edge_batch, mp_ptr, mp_eids):
        src, dst = edge_index[0, edge_batch], edge_index[1, edge_batch]

        # Directed payments only (no reverse edges) — ~2× fewer VQC edges.
        # Sample neighbours from the pre-sliced MP graph (train / train+val / full).
        seeds = torch.cat([src, dst])
        sub_index, sub_attr = sample_subgraph(
            seeds, mp_edge_index, mp_edge_attr, self.fanouts, mp_ptr, mp_eids
        )

        used, mapper, sub_local = compact_nodes(sub_index, seeds)
        
        src_l, dst_l = mapper[src], mapper[dst]

        h = F.relu(self.node_enc(x[used]))
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        
        # One quantum pass; both hops reuse the same α_ij
        alpha = self.quantum_alphas(h, sub_local, sub_attr)
        for conv in self.convs:
            h = conv(h, sub_local, alpha)
            h = F.dropout(h, p=self.dropout, training=self.training)

        ea = edge_attr[edge_batch]
        return self.classifier(torch.cat([h[src_l], h[dst_l], ea], dim=-1)).squeeze(-1)



def message_graph(split):
    """Split-prefix MP: val uses train edges; test uses train+val. Query edges stay in edge_batch."""
    if split == "train":
        return train_edge_index, train_edge_attr, train_mp_ptr, train_mp_eids
    if split == "val":
        return train_edge_index, train_edge_attr, train_mp_ptr, train_mp_eids
    if split == "test":
        return val_edge_index, val_edge_attr, val_mp_ptr, val_mp_eids
    raise ValueError(f"unknown split {split!r}")


def predict_edges(net, idx, split="val"):
    net.eval()
    mp_ei, mp_ea, ptr, eids = message_graph(split)
    logits_all, trues = [], []
    with torch.no_grad():
        for start in range(0, idx.size(0), BATCH_SIZE):
            batch = idx[start:start + BATCH_SIZE]
            logits = net(
                node_features, mp_ei, mp_ea, edge_index, edge_attr, batch, ptr, eids
            )
            logits_all.append(logits.cpu())
            trues.append(edge_labels[batch].cpu())
    return torch.cat(logits_all), torch.cat(trues)

print("Model Setup")

def pick_device(qdev):
    name = str(getattr(qdev, "name", qdev)).lower()
    if "gpu" in name and torch.cuda.is_available():
        return torch.device("cuda")
    # lightning.qubit / default.qubit are CPU simulators — keep tensors on CPU
    return torch.device("cpu")

device = pick_device(dev)
print("Using device:", device)


def to_torch_device(module, device):
    """nn.Module.to() can leave PennyLane TorchLayer weights on CPU."""
    module.to(device)
    for m in module.modules():
        if isinstance(m, qml.qnn.TorchLayer):
            m.to(device)
    return module



edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
edge_labels = edge_labels.to(device)
train_mask = train_mask.to(device)
val_mask = val_mask.to(device)
test_mask = test_mask.to(device)
num_nodes = len(node_mapping)
BATCH_SIZE = 512
MAX_EVAL_EDGES = 8192  # subsample val for faster epoch-end metrics
VAL_EVERY = 2  # validate every N epochs; full test runs once after training

train_idx = train_mask.nonzero(as_tuple=False).view(-1)
val_idx = val_mask.nonzero(as_tuple=False).view(-1)
test_idx = test_mask.nonzero(as_tuple=False).view(-1)
node_features = node_features.to(device)
train_edge_index = train_edge_index.to(device)
train_edge_attr = train_edge_attr.to(device)
val_edge_index = val_edge_index.to(device)
val_edge_attr = val_edge_attr.to(device)
print(
    f"QGNN node_attr_dim={node_features.shape[1]} "
    f"(train-only log1p in/out degree) | FANOUTS={FANOUTS} | MAX_Q_EDGES={MAX_Q_EDGES}"
)
train_mp_ptr, train_mp_eids = build_incident_csr(train_edge_index, num_nodes)
val_mp_ptr, val_mp_eids = build_incident_csr(val_edge_index, num_nodes)


## Hyperparameters

Training supports **Adam**, **AdamW**, and **SGD**. Set `OPTIMIZER` in the next cell to pick one, or leave it as `None` to use the name in `best_params.json`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


SUPPORTED_OPTIMIZERS = ("Adam", "AdamW", "SGD")
# Set to "Adam", "AdamW", or "SGD" to override best_params.json. None keeps the file's choice.
OPTIMIZER = None


def make_optimizer(params, parameters):
    """Adam, AdamW, or SGD. Name is case-insensitive; extra kwargs are optimizer-specific."""
    raw = params.get("optimizer", "Adam")
    resolved = {k.lower(): k for k in SUPPORTED_OPTIMIZERS}.get(str(raw).strip().lower())
    if resolved is None:
        raise ValueError(
            f"optimizer must be one of {SUPPORTED_OPTIMIZERS}, got {raw!r}"
        )

    kwargs = {"lr": params["lr"], "weight_decay": params["weight_decay"]}
    if resolved == "SGD":
        kwargs["momentum"] = params.get("momentum", 0.9)
        kwargs["nesterov"] = params.get("nesterov", False)
        opt = optim.SGD(parameters, **kwargs)
        extra = f" | momentum={kwargs['momentum']} | nesterov={kwargs['nesterov']}"
    else:
        if "betas" in params:
            kwargs["betas"] = tuple(params["betas"])
        if "eps" in params:
            kwargs["eps"] = params["eps"]
        opt = (optim.AdamW if resolved == "AdamW" else optim.Adam)(parameters, **kwargs)
        extra = ""

    print(
        f"Using {resolved} | lr={kwargs['lr']} | weight_decay={kwargs['weight_decay']}{extra}"
    )
    return opt


if BEST_PARAMS_PATH is not None:
    with BEST_PARAMS_PATH.open() as f:
        best_params = json.load(f)
    print(f"Loaded {BEST_PARAMS_PATH}:", best_params)
else:
    best_params = dict(_FALLBACK_BEST_PARAMS)
    print("best_params.json not found — using fallback:", best_params)

if OPTIMIZER is not None:
    best_params["optimizer"] = OPTIMIZER
    print(f"OPTIMIZER override -> {OPTIMIZER}")

print("\n--- Starting Phase 5b: Training Final QGNN ---")


def _wandb_login():
    """Authenticate W&B. On Kaggle, attach secret WANDB_API_KEY (from https://wandb.ai/authorize)."""
    key = os.environ.get("WANDB_API_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception:
            key = None
    if key:
        wandb.login(key=key)
        return True
    print(
        "No W&B API key — logging disabled (training continues). "
        "On Kaggle: Add-ons → Secrets → add WANDB_API_KEY "
        "(from https://wandb.ai/authorize) and attach it to this notebook."
    )
    return False


USE_WANDB = _wandb_login()

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="lachlan-bruce-curtin-university",
    # Set the wandb project where this run will be logged.
    project="QDetect QGNN",
    mode="online" if USE_WANDB else "disabled",
    # Track hyperparameters and run metadata.
    config={
        **best_params,
        "epochs": 30,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "n_qubits": n_qubits,
        "pos_weight_train": float(pos_weight_train),
    },
)
wandb.define_metric("epoch")
wandb.define_metric("train/*", step_metric="epoch")
wandb.define_metric("val/*", step_metric="epoch")
loss_epochs = []
loss_train_hist = []
loss_val_hist = []
auprc_hist = []





model = to_torch_device(QGNN(
    num_nodes=num_nodes,
    node_attr_dim=node_features.shape[1],
    edge_attr_dim=train_edge_attr.shape[1],
    hidden_dim=best_params["hidden_dim"],
    dropout=best_params.get("dropout", 0.3),
), device)
optimizer = make_optimizer(best_params, model.parameters())
pos_weight_tensor = torch.tensor(
    [float(pos_weight_train) * best_params["pos_weight_factor"]], device=device
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
l1_lambda = float(best_params.get("l1_lambda", 1e-7))
print(f"Regulariser: L1={l1_lambda:.2e} (Laplacian skipped — QGNN uses sampled subgraphs, not full-graph encode)")
epochs = 30
patience = int(PATIENCE)  # epochs without val-AUPRC improvement; not tied to VAL_EVERY
best_val_auprc = -1.0
best_val_loss = float("inf")
best_state = None
last_improve_epoch = 0


def subsample_eval_idx(idx):
    if idx.numel() > MAX_EVAL_EDGES:
        return idx[torch.randperm(idx.size(0), device=device)[:MAX_EVAL_EDGES]]
    return idx


def metrics_from_logits(logits, true):
    """BCE + AUPRC from one forward (same edges and the same sampled subgraphs)."""
    loss_device = criterion.pos_weight.device
    loss = criterion(
        logits.to(loss_device),
        true.to(device=loss_device, dtype=logits.dtype),
    ).item()
    prob = torch.sigmoid(logits).detach().cpu().numpy()
    y = true.detach().cpu().numpy()
    return loss, average_precision_score(y, prob), y, prob


def split_stats(y_true, y_prob, threshold=0.5):
    """AUPRC is threshold-free; F1/acc/prec/rec use a fixed 0.5 cutoff (same as 750k)."""
    y_bin = (y_prob >= threshold).astype(int)
    return (
        float(average_precision_score(y_true, y_prob)),
        float(accuracy_score(y_true, y_bin)),
        float(precision_score(y_true, y_bin, zero_division=0)),
        float(recall_score(y_true, y_bin, zero_division=0)),
        float(f1_score(y_true, y_bin, zero_division=0)),
    )


def eval_split_metrics(net, idx, split):
    """BCE loss + 750k-style ranking/threshold metrics from one predict_edges forward."""
    logits, true = predict_edges(net, idx, split=split)
    loss, _, y, prob = metrics_from_logits(logits, true)
    auprc, acc, prec, rec, f1 = split_stats(y, prob)
    return {
        "loss": loss,
        "auprc": auprc,
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1,
    }


def eval_loss_auprc(net, idx, split):
    m = eval_split_metrics(net, idx, split)
    return m["loss"], m["auprc"]


def choose_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = (2 * prec * rec) / np.clip(prec + rec, 1e-12, None)
    f1_thr = f1[:-1] if len(f1) == len(thr) + 1 else f1
    if len(f1_thr) == 0:
        return 0.5, 0.0
    i = int(np.nanargmax(f1_thr))
    return float(thr[i]), float(f1_thr[i])


def specificity_score(y_true, y_bin):
    tn, fp, fn, tp = confusion_matrix(y_true, y_bin, labels=[0, 1]).ravel()
    return float(tn / max(tn + fp, 1))


def print_split_metrics(name, y_true, y_prob, threshold, loss=None):
    y_bin = (y_prob >= threshold).astype(int)
    auprc = average_precision_score(y_true, y_prob)
    auroc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_bin)
    prec = precision_score(y_true, y_bin, zero_division=0)
    rec = recall_score(y_true, y_bin, zero_division=0)
    f1 = f1_score(y_true, y_bin, zero_division=0)
    spec = specificity_score(y_true, y_bin)
    base = float(np.mean(y_true))
    print("=" * 60)
    print(f" {name}")
    print("=" * 60)
    print(f"  n:            {len(y_true):,}")
    print(f"  base rate:    {base:.4%}   (dummy AUPRC ≈ this)")
    print(f"  threshold:    {threshold:.4f}  (max-F1 on val, frozen)")
    if loss is not None:
        print(f"  loss:         {loss:.4f}")
    print(f"  AUPRC:        {auprc:.4f}")
    print(f"  AUROC:        {auroc:.4f}")
    print(f"  Accuracy:     {acc:.4f}")
    print(f"  Precision:    {prec:.4f}")
    print(f"  Recall:       {rec:.4f}")
    print(f"  F1:           {f1:.4f}")
    print(f"  Specificity:  {spec:.4f}")
    print()
    print(classification_report(y_true, y_bin, digits=4, zero_division=0, target_names=["legit", "laundering"]))
    cm = confusion_matrix(y_true, y_bin, labels=[0, 1])
    print("Confusion matrix [[TN FP] [FN TP]]:")
    print(cm)
    return {
        "auprc": auprc, "auroc": auroc, "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1, "specificity": spec, "threshold": threshold,
        "base_rate": base, "cm": cm, "loss": loss, "n": int(len(y_true)),
    }


def ranking_metrics(net, idx, split="test"):
    logits, true = predict_edges(net, idx, split=split)
    loss, auprc, y, prob = metrics_from_logits(logits, true)
    return loss, y, prob


total_batches = math.ceil(train_idx.size(0) / BATCH_SIZE)
train_eval_idx = subsample_eval_idx(train_idx)
val_eval_idx = subsample_eval_idx(val_idx)


for epoch in range(1, epochs + 1):
    model.train()
    perm = train_idx[torch.randperm(train_idx.size(0), device=device)]
    total_loss = 0.0
    n_batches = 0

    mp_ei, mp_ea, ptr, eids = message_graph("train")
    for start in range(0, perm.size(0), BATCH_SIZE):
        batch = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        _t_mark = time.time()
        logits = model(
            node_features, mp_ei, mp_ea, edge_index, edge_attr, batch, ptr, eids
        )
        t_fwd = time.time() - _t_mark
        bce = criterion(logits.to(device), edge_labels[batch].to(device))
        l1 = sum(p.abs().sum() for p in model.parameters())
        loss = bce + l1_lambda * l1
        _t_mark = time.time()
        loss.backward()
        t_bwd = time.time() - _t_mark
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        if n_batches == 1 or n_batches % 50 == 0 or n_batches == total_batches:
            print(
                f"  [train] epoch {epoch}/{epochs} | batch {n_batches}/{total_batches} | "
                f"loss={loss.item():.4f} | fwd={t_fwd:.1f}s bwd={t_bwd:.1f}s",
                flush=True,
            )

    train_batch_loss = total_loss / max(n_batches, 1)

    # Every VAL_EVERY epochs: full 750k-style train/val suite @ thr=0.5; wandb only then.
    val_loss = val_auprc = None
    if epoch % VAL_EVERY == 0 or epoch == epochs:
        model.eval()
        with torch.no_grad():
            train_m = eval_split_metrics(model, train_eval_idx, split="train")
            val_m = eval_split_metrics(model, val_eval_idx, split="val")
        train_loss = train_m["loss"]
        val_loss = val_m["loss"]
        val_auprc = val_m["auprc"]
        log = {
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/auprc": train_m["auprc"],
            "val/auprc": val_auprc,
            "train/acc": train_m["acc"],
            "val/acc": val_m["acc"],
            "train/prec": train_m["prec"],
            "val/prec": val_m["prec"],
            "train/rec": train_m["rec"],
            "val/rec": val_m["rec"],
            "train/f1": train_m["f1"],
            "val/f1": val_m["f1"],
        }
        loss_epochs.append(epoch)
        loss_train_hist.append(train_loss)
        loss_val_hist.append(val_loss)
        auprc_hist.append(val_auprc)
        if USE_WANDB:
            log["loss_curves"] = wandb.plot.line_series(
                xs=loss_epochs,
                ys=[loss_train_hist, loss_val_hist],
                keys=["train", "val"],
                title="Train / Val Loss",
                xname="epoch",
            )
            wandb.log(log)
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            last_improve_epoch = epoch
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"train loss={train_loss:.4f} AUPRC={train_m['auprc']:.4f} F1={train_m['f1']:.4f} | "
            f"val loss={val_loss:.4f} AUPRC={val_auprc:.4f} F1={val_m['f1']:.4f} "
            f"(best={best_val_auprc:.4f}) | batch_loss={train_batch_loss:.4f}",
            flush=True,
        )
    else:
        print(
            f"Epoch {epoch:03d}/{epochs} | train batch loss={train_batch_loss:.4f}",
            flush=True,
        )

    if last_improve_epoch > 0 and epoch - last_improve_epoch >= patience:
        print(
            f"Early stopping at epoch {epoch} "
            f"(best val AUPRC {best_val_auprc:.4f}, no improve for {patience} epochs)",
            flush=True,
        )
        break


if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best val-AUPRC weights (AUPRC={best_val_auprc:.4f}, val loss={best_val_loss:.4f})")

# Test once, after loading the val-AUPRC checkpoint. Val-tuned F1 threshold frozen for test.
val_logits, val_true_t = predict_edges(model, subsample_eval_idx(val_idx), split="val")
val_eval_loss, val_eval_auprc, val_y, val_prob = metrics_from_logits(val_logits, val_true_t)
decision_threshold, val_best_f1 = choose_threshold(val_y, val_prob)
print(f"Val-tuned threshold: {decision_threshold:.4f} (val F1={val_best_f1:.4f})")
val_metrics = print_split_metrics("VALIDATION (MP=train, query=val)", val_y, val_prob, decision_threshold, val_eval_loss)

test_loss, test_y, test_prob = ranking_metrics(model, test_idx, split="test")
test_metrics = print_split_metrics(
    "OUT-OF-TIME TEST (MP=train+val, query=test)",
    test_y, test_prob, decision_threshold, test_loss,
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
p_prec, p_rec, _ = precision_recall_curve(test_y, test_prob)
axes[0].plot(p_rec, p_prec, color="crimson", label=f"AUPRC={test_metrics['auprc']:.3f}")
axes[0].axhline(test_metrics["base_rate"], color="steelblue", linestyle="--", label=f"dummy={test_metrics['base_rate']:.3f}")
axes[0].set_title("Test precision-recall")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precision")
axes[0].legend()

fpr, tpr, _ = roc_curve(test_y, test_prob)
axes[1].plot(fpr, tpr, color="crimson", label=f"AUROC={test_metrics['auroc']:.3f}")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1].set_title("Test ROC")
axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("true positive rate")
axes[1].legend()

cm = test_metrics["cm"]
tn, fp, fn, tp = cm.ravel()
im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_title(f"Test confusion matrix @ thr={decision_threshold:.3f}")
axes[2].set_xticks([0, 1], ["pred legit", "pred launder"])
axes[2].set_yticks([0, 1], ["true legit", "true launder"])
cell_names = [["TN", "FP"], ["FN", "TP"]]
for (i, j), v in np.ndenumerate(cm):
    axes[2].text(j, i, f"{cell_names[i][j]}\n{v:,}", ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

print("\n--- Leak audit ---")
print(f"  SEED_POLICY          = {SEED_POLICY}")
print(f"  MP_POLICY            = {MP_POLICY}")
print(f"  node_attr_dim        = {node_features.shape[1]} (train log1p in/out degree)")
print(f"  test AUPRC           = {test_metrics['auprc']:.4f}  (target {TARGET_AUPRC[0]:.2f}–{TARGET_AUPRC[1]:.2f})")
if test_metrics["auprc"] > LEAK_AUPRC:
    print("  TEST AUPRC > 0.82 — treat as a leak warning, not a modelling win.")
elif TARGET_AUPRC[0] <= test_metrics["auprc"] <= TARGET_AUPRC[1]:
    print("  Test AUPRC is inside the expected unleaked band.")
else:
    print("  Test AUPRC is below the 0.70 floor — do not relax leak constraints to chase it.")

print(
    f"Held-out val | loss={val_eval_loss:.4f} | AUPRC={val_eval_auprc:.4f} | "
    f"train batch loss={train_batch_loss:.4f}"
)

wandb.summary["best_val_auprc"] = best_val_auprc
wandb.summary["best_val_loss"] = best_val_loss
wandb.summary["final_train_loss"] = train_batch_loss
wandb.summary["final_val_loss"] = val_eval_loss
wandb.summary["final_val_auprc"] = val_eval_auprc
wandb.summary["decision_threshold"] = decision_threshold
wandb.summary["final_test_loss"] = test_metrics["loss"]
wandb.summary["final_test_auprc"] = test_metrics["auprc"]
wandb.summary["final_test_precision"] = test_metrics["precision"]
wandb.summary["final_test_recall"] = test_metrics["recall"]
wandb.summary["final_test_f1"] = test_metrics["f1"]
wandb.summary["test_base_rate"] = test_metrics["base_rate"]

run.finish()

state = best_state if best_state is not None else model.state_dict()
state = {k: v.detach().cpu().contiguous() for k, v in state.items()}
if not hasattr(t0, "isoformat"):
    raise TypeError(
        f"graph t0 was overwritten before serving save (got {type(t0).__name__}={t0!r}); "
        "do not reuse the name t0 for time.time() timers"
    )
t0_json = t0.isoformat()
serving = {
    "model_state": state,
    "model_state_dict": state,
    "architecture": "QGNN",
    "hidden_dim": int(best_params["hidden_dim"]),
    "n_qubits": int(n_qubits),
    "node_attr_dim": int(node_features.shape[1]),
    "edge_attr_dim": int(train_edge_attr.shape[1]),
    "dropout": float(best_params.get("dropout", 0.3)),
    "fanouts": tuple(FANOUTS),
    "scaler": scaler,
    "cat_encoder": cat_encoder,
    "t0": t0_json,
    "clip": float(globals().get("clip", 10.0)),
    "cont_cols": list(CONT_COLS),
    "cat_cols": list(CAT_COLS),
    "bin_cols": list(BIN_COLS),
    "feature_names": list(feature_names),
    "rates": dict(RATES),
    "decision_threshold": float(decision_threshold),
}
torch.save(serving, SERVING_PATH)
print(f"Saved serving bundle to {SERVING_PATH.resolve()}")

# Graph-aligned test CSV (same Timestamp sort as graph build, then test_mask).
table = data.copy()
if "Timestamp" not in table.columns and "ts" in table.columns:
    table["Timestamp"] = pd.to_datetime(table["ts"], errors="coerce")
table = table.sort_values("Timestamp").reset_index(drop=True)
test_mask_np = test_mask.detach().cpu().numpy()
if len(table) != int(test_mask.numel()):
    table = df_sampled.sort("ts").to_pandas()
    if "Timestamp" not in table.columns and "ts" in table.columns:
        table["Timestamp"] = pd.to_datetime(table["ts"], errors="coerce")
    table = table.sort_values("Timestamp").reset_index(drop=True)
if len(table) != int(test_mask.numel()):
    raise ValueError(
        f"Test CSV table length {len(table):,} != graph edges {int(test_mask.numel()):,}; "
        "cannot align rows with test_mask"
    )
test_df = table.loc[test_mask_np].copy()
test_df["risk_score"] = np.asarray(test_prob, dtype=np.float64)
test_df.to_csv(TEST_CSV_PATH, index=False)
print(f"Wrote {len(test_df):,} test edges to {TEST_CSV_PATH.resolve()}")
